In [ ]:
# NBVAL_SKIP
from jax import config
#config.update("jax_enable_x64", True)
config.update('jax_num_cpu_devices', 2)

In [ ]:
#NBVAL_SKIP
import os

# Tell XLA to fake 2 host CPU devices
#os.environ['XLA_FLAGS'] = '--xla_force_host_platform_device_count=3'

# Only make GPU 0 and GPU 1 visible to JAX:
os.environ['CUDA_VISIBLE_DEVICES'] = '1,3,4,5,6,7,8,9'

#os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]   = "false"

import jax

# Now JAX will list two CpuDevice entries
print(jax.devices())
# → [CpuDevice(id=0), CpuDevice(id=1)]

In [ ]:
# NBVAL_SKIP
#import os
#  os.environ['SPS_HOME'] = '/mnt/storage/annalena_data/sps_fsps'
#os.environ['SPS_HOME'] = '/home/annalena/sps_fsps'
#os.environ['SPS_HOME'] = '/Users/annalena/Documents/GitHub/fsps'
os.environ['SPS_HOME'] = '/export/home/aschaibl/fsps'
#os.environ['SPS_HOME'] = '/home/annalena_data/sps_fsps'

# RUBIX pipeline

RUBIX is designed as a linear pipeline, where the individual functions are called and constructed as a pipeline. This allows as to execude the whole data transformation from a cosmological hydrodynamical simulation of a galaxy to an IFU cube in two lines of code. This notebook shows, how to execute the pipeline. To see, how the pipeline is execuded in small individual steps per individual function, we refer to the notebook `rubix_pipeline_stepwise.ipynb`.

## How to use the Pipeline
1) Define a `config`
2) Setup the `pipeline yaml`
3) Run the RUBIX pipeline
4) Do science with the mock-data

## Step 1: Config



In [ ]:
#NBVAL_SKIP
import matplotlib.pyplot as plt
from rubix.core.pipeline import RubixPipeline 
import os

config_TNG = {
    "pipeline":{"name": "calc_ifu_memory"},
    
    "logger": {
        "log_level": "DEBUG",
        "log_file_path": None,
        "format": "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    },
    "data": {
        "name": "IllustrisAPI",
        "args": {
            "api_key": os.environ.get("ILLUSTRIS_API_KEY"),
            "particle_type": ["stars"],
            "simulation": "TNG50-1",
            "snapshot": 99,
            "save_data_path": "data",
        },
        
        "load_galaxy_args": {
        "id": 11,
        "reuse": True,
        },
        
        "subset": {
            "use_subset": True,
            "subset_size": 1,
        },
    },
    "simulation": {
        "name": "IllustrisTNG",
        "args": {
            "path": "data/galaxy-id-11.hdf5",
        },
    
    },
    "output_path": "output",

    "telescope":
        {"name": "MUSE",
         "psf": {"name": "gaussian", "size": 5, "sigma": 0.6},
         "lsf": {"sigma": 0.5},
         "noise": {"signal_to_noise": 100,"noise_distribution": "normal"},},
    "cosmology":
        {"name": "PLANCK15"},
        
    "galaxy":
        {"dist_z": 0.1,
         "rotation": {"type": "edge-on"},
        },
        
    "ssp": {
        "template": {
            "name": "FSPS", #"Mastar_CB19_SLOG_1_5"
        },
        "dust": {
                "extinction_model": "Cardelli89",
                "dust_to_gas_ratio": 0.01,
                "dust_to_metals_ratio": 0.4,
                "dust_grain_density": 3.5,
                "Rv": 3.1,
            },
    },        
}

## Step 3: Run the pipeline

After defining the `config` and the `pipeline_config` you can simply run the whole pipeline by these two lines of code.

In [ ]:
#NBVAL_SKIP
pipe = RubixPipeline(config_TNG)

In [ ]:
#NBVAL_SKIP
import jax.numpy as jnp

inputdata = pipe.prepare_data()
#inputdata = pipe.prepare_data()
coords = inputdata.stars.coords
vel = inputdata.stars.velocity
mass = inputdata.stars.mass
age = inputdata.stars.age
met = inputdata.stars.metallicity
factor = 1
inputdata.stars.coords = jnp.concatenate([coords]*factor, axis=0)
inputdata.stars.velocity = jnp.concatenate([vel]*factor, axis=0)
inputdata.stars.mass = jnp.concatenate([mass]*factor, axis=0)
inputdata.stars.age = jnp.concatenate([age]*factor, axis=0)
inputdata.stars.metallicity = jnp.concatenate([met]*factor, axis=0)
inputdata.stars.coords.shape

In [ ]:
#NBVAL_SKIP

rubixdata = pipe.run_sharded(inputdata)

In [ ]:
rubixdata = pipe.run_sharded(inputdata)

In [ ]:
import jax.numpy as jnp
gpu_number = jnp.array([1, 2, 3, 4, 5, 6, 7])
time_on_compgpu4_5e5mal2 = jnp.array([274.27, 152.38, 108.70, 88.38, 88.97, 71.85, 62.91])
time_on_compgpu4_5e5mal1 = jnp.array([151.12, 77.44, 63.81, 50.88, 48.34, 48.1, 41.60])

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(5,3))
plt.plot(gpu_number, time_on_compgpu4_5e5mal2, marker='o', label='1e6 particles')
plt.plot(gpu_number, time_on_compgpu4_5e5mal1, marker='o', label='5e5 particles')
plt.xlabel('Number of GPUs')
plt.ylabel('Time in seconds on RTX 2080ti')
plt.legend()

In [ ]:
import jax.numpy as jnp
particle_number = jnp.array([1, 10, 100, 1e3, 1e4, 1e5, 5e5, 5e5*2, 5e5*20, 5e5*50, 5e5*100, 5e5*150, 5e5*200, 5e5*300, 5e5*400])
time_on_mac_2cpu = jnp.array([2.14, 2.14, 2.24, 2.2, 2.2 ,2.15, 2.34, 2.26, 2.50, 3.78, 16.88, 38.92, 56.29, 72.27, 86.98]) #seconds
particle_number_gpu = jnp.array([1, 10, 100, 1e3, 1e4, 1e5, 5e5, 5e5*2, 5e5*4, 5e5*20])
time_on_compgpu4_2gpu = jnp.array([18.01, 18.64, 18.44, 18.58, 20.43, 31.14, 84.95, 138.29, 255.22, 1182.35])
time_on_compgpu4_4gpu = jnp.array([19.11, 19.18, 19.69, 19.26, 20.74, 27.97, 59.56, 89.58, 142.98, 707.86])
time_on_compgpu4_6gpu = jnp.array([20.14, 20.22, 20.34, 20.85, 20.48, 25.59, 47.19, 76.89, 122.12, 500.78])

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 3))
plt.plot(particle_number_gpu, time_on_compgpu4_2gpu, marker='o', label='2 GPUs')
plt.plot(particle_number_gpu, time_on_compgpu4_4gpu, marker='o', label='4 GPUs')
plt.plot(particle_number_gpu, time_on_compgpu4_6gpu, marker='o', label='6 GPUs')
plt.xlabel('Number of particles')
plt.ylabel('Time in seconds on RTX 2080ti')
plt.xscale('log')
plt.yscale('log')
plt.legend()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 3))
plt.plot(particle_number_gpu, time_on_compgpu4_2gpu, marker='o', label='2 GPUs')
plt.plot(particle_number_gpu, time_on_compgpu4_4gpu, marker='o', label='4 GPUs')
plt.plot(particle_number_gpu, time_on_compgpu4_6gpu, marker='o', label='6 GPUs')
plt.plot(particle_number, time_on_mac_2cpu, marker='x', label='MacBook Pro 2 CPUs')
plt.xlabel('Number of particles')
plt.ylabel('Time in seconds')
plt.xscale('log')
plt.yscale('log')
plt.legend()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 3))
plt.plot(particle_number_gpu, time_on_compgpu4_2gpu/particle_number_gpu, marker='o', label='2 GPUs')
plt.plot(particle_number_gpu, time_on_compgpu4_4gpu/particle_number_gpu, marker='o', label='4 GPUs')
plt.plot(particle_number_gpu, time_on_compgpu4_6gpu/particle_number_gpu, marker='o', label='6 GPUs')
plt.xlabel('Number of particles')
plt.ylabel('Time per particle in seconds')
plt.xscale('log')
plt.yscale('log')
plt.legend()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 3))
plt.plot(particle_number, time_on_mac_2cpu, marker='o', linestyle='-')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Number of particles')
plt.ylabel('Time in seconds on M1')
#plt.title('Scaling of Rubix Pipeline with Number of Particles')

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 3))
plt.plot(particle_number, time_on_mac_2cpu/particle_number, marker='o', linestyle='-')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Number of particles')
plt.ylabel('Time per particle in seconds')
#plt.title('Scaling of Rubix Pipeline with Number of Particles')